# 11_Full_Pipeline_V11

Clean end-to-end rope-flow pipeline with:
1. Labeled training (JSON segment midpoint windows)
2. Pseudo-labeling on unlabeled cycles (confidence threshold)
3. Retraining on labeled + high-confidence pseudo-labeled samples

Feature toggles are configurable at CFG level (Fourier / Biomech / Combined).

## 1) Imports and configuration

In [32]:
import os
import glob
import json

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, coherence

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

np.random.seed(42)

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PROCESSED = os.path.join(ROOT, 'data', 'processed')
LABELED_RAW_BASE = os.path.join(ROOT, 'data', 'raw', 'new-labeled-sessions')

CFG = {
    'FS': 50.0,
    'WINDOW': 40,
    'PEAK_PROM_DEGS': 50.0,
    'PEAK_MIN_DEGS': 50.0,
    'PEAK_SAVGOL_WINDOW': 15,
    'PEAK_SAVGOL_POLY': 3,
    'PEAK_MIN_PERIOD_S': 0.2,
    'MERGE_GAP_S': 0.20,
    'FOURIER_K': 8,
    'PSEUDO_CONF_THRESHOLD': 0.80,
    'PCA_VAR': 0.99,
    'NN_EPOCHS': 80,
    'NN_BATCH_SIZE': 32,
    # Feature toggles (set these to try separately or combined)
    'USE_RAW_FLAT_FEATURES': True,
    'USE_FOURIER_FEATURES': False,
    'USE_BIOMECH_FEATURES': True,
    'USE_COHERENCE_FEATURES': True,
    # Pseudo-label quality gates
    'USE_CLUSTER_PURITY_GATE': True,
    'CLUSTER_K': 10,
    'CLUSTER_PURITY_THRESHOLD': 0.80,
    'REQUIRE_RF_CLUSTER_AGREEMENT': True,
    'MIN_LR_SAMPLES_PER_CLUSTER': 5,
    'SUPERVISED_CLASSES': [
        'dragon_roll',
        'underhand_right',
        'underhand_left',
        'overhand_left',
        'overhand_right',
        'sneak_underhand_left',
        'sneak_underhand_right',
        'sneak_overhand_left',
        'sneak_overhand_right',
        'clockwise',
        'counter_clockwise',
        'idle',
    ],
}

CLASS_NAMES = list(CFG['SUPERVISED_CLASSES'])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

BIO_FEAT_NAMES = ['gz_mean_d0']

## 2) Data and feature helpers

In [33]:
LABEL_ALIAS_MAP = {
    'bf': 'dragon_roll', 'bf2': 'dragon_roll', 'fb': 'dragon_roll', 'fb2': 'dragon_roll',
    'dragon roll': 'dragon_roll', 'dragon_roll': 'dragon_roll',
    'ur': 'underhand_right', 'ur0': 'underhand_right',
    'underhand right': 'underhand_right', 'underhand': 'underhand_right',
    'ul': 'underhand_left', 'ul0': 'underhand_left', 'underhand left': 'underhand_left',
    'ol': 'overhand_left', 'ol0': 'overhand_left', 'ol2': 'overhand_left',
    'overhand left': 'overhand_left', 'overhand': 'overhand_left',
    'or': 'overhand_right', 'or2': 'overhand_right', 'or3': 'overhand_right',
    'overhand right': 'overhand_right',
    'usl': 'sneak_underhand_left', 'sneak underhand left': 'sneak_underhand_left',
    'usr': 'sneak_underhand_right', 'sneak underhand right': 'sneak_underhand_right',
    'osl': 'sneak_overhand_left', 'sneak overhand left': 'sneak_overhand_left',
    'osr': 'sneak_overhand_right', 'sneak overhand right': 'sneak_overhand_right',
    'cw': 'clockwise', 'clockwise': 'clockwise',
    'ccw': 'counter_clockwise', 'counter clockwise': 'counter_clockwise',
    'idle': 'idle', 'idle3': 'idle', 'no movement': 'idle',
    'excluded': 'excluded', 'vq': 'excluded', 'vq5': 'excluded', 'vq15': 'excluded', 'vq16': 'excluded',
}

def normalize_label_key(label):
    s = str(label).strip().lower().replace('_', ' ').replace('-', ' ')
    return ' '.join(s.split())

def canonicalize_label(label):
    key = normalize_label_key(label)
    if key in LABEL_ALIAS_MAP:
        return LABEL_ALIAS_MAP[key]
    for sep in ('/', '|'):
        if sep in key:
            first = normalize_label_key(key.split(sep)[0].strip())
            return LABEL_ALIAS_MAP.get(first, first)
    return key

def map_to_supervised_class(raw_label):
    lab = canonicalize_label(raw_label)
    if lab == 'excluded':
        return None
    return lab if lab in CLASS_TO_IDX else None

def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def discover_sessions(processed_dir):
    entries = []
    for d0 in sorted(glob.glob(os.path.join(processed_dir, '*_device0_processed.csv'))):
        d1 = d0.replace('_device0_', '_device1_')
        if os.path.isfile(d1):
            sid = os.path.basename(d0).replace('_device0_processed.csv', '')
            entries.append((d0, d1, sid))
    return entries

def load_annotations(session_id):
    session_dir = os.path.join(LABELED_RAW_BASE, session_id)
    if not os.path.isdir(session_dir):
        return None

    for name in ('labels_corrected.json', 'labels.json', 'labels_vad.json'):
        path = os.path.join(session_dir, name)
        if not os.path.isfile(path):
            continue
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        segs = data.get('segments', []) if isinstance(data, dict) else data
        clean = []
        if isinstance(segs, list):
            for s in segs:
                if not isinstance(s, dict):
                    continue
                t0 = safe_float(s.get('start'))
                t1 = safe_float(s.get('end'))
                lab = s.get('label')
                if t0 is None or t1 is None or lab is None or t1 <= t0:
                    continue
                clean.append((t0, t1, str(lab)))

        if clean:
            return {'json_path': path, 'segments': clean}
    return None

def maybe_align_annotation_time(segments, csv_t_min, csv_t_max):
    all_times = [t for t0, t1, _ in segments for t in (t0, t1)]
    if not all_times:
        return segments
    json_t_min = min(all_times)
    json_t_max = max(all_times)
    if json_t_min > csv_t_max or json_t_max < csv_t_min:
        offset = json_t_min - csv_t_min
        return [(t0 - offset, t1 - offset, lab) for t0, t1, lab in segments]
    return segments

def load_session_signals(d0_path, d1_path):
    d0 = pd.read_csv(d0_path)
    d1 = pd.read_csv(d1_path)
    t0 = d0['timestamp_ms'].to_numpy(dtype=np.float64) / 1000.0
    t1 = d1['timestamp_ms'].to_numpy(dtype=np.float64) / 1000.0
    ch0 = d0[['ax_w', 'ay_w', 'az_w', 'gx', 'gy', 'gz']].to_numpy(dtype=np.float32)
    ch1 = d1[['ax_w', 'ay_w', 'az_w', 'gx', 'gy', 'gz']].to_numpy(dtype=np.float32)
    return t0, ch0, t1, ch1

def smooth_mag_deg(omega_deg, cfg):
    mag = np.linalg.norm(omega_deg, axis=1)
    n = len(mag)
    if n < 7:
        return mag
    win = int(cfg['PEAK_SAVGOL_WINDOW'])
    if win % 2 == 0:
        win += 1
    max_odd = n if n % 2 == 1 else n - 1
    win = max(5, min(win, max_odd))
    poly = max(1, min(int(cfg['PEAK_SAVGOL_POLY']), win - 2))
    return savgol_filter(mag, window_length=win, polyorder=poly, mode='interp')

def detect_cycle_peaks(omega_deg, fs, cfg):
    mag_s = smooth_mag_deg(omega_deg, cfg)
    peaks, _ = find_peaks(
        mag_s,
        distance=max(1, int(cfg['PEAK_MIN_PERIOD_S'] * fs)),
        prominence=cfg['PEAK_PROM_DEGS'],
    )
    return np.array([int(p) for p in peaks if mag_s[p] >= cfg['PEAK_MIN_DEGS']], dtype=int)

def merge_device_peaks_pairs(peaks0, peaks1, fs, gap_s):
    tagged = [(p / fs, p, 'D0') for p in peaks0] + [(p / fs, p, 'D1') for p in peaks1]
    if not tagged:
        return []
    tagged.sort(key=lambda x: x[0])
    times = [x[0] for x in tagged]

    accepted = [0]
    for i in range(1, len(times)):
        if times[i] - times[accepted[-1]] > gap_s:
            accepted.append(i)

    groups = [{} for _ in accepted]
    a_idx = 0
    for i, (_, peak_idx, src) in enumerate(tagged):
        if a_idx + 1 < len(accepted) and i >= accepted[a_idx + 1]:
            a_idx += 1
        groups[a_idx].setdefault(src, peak_idx)
    return [(g.get('D0'), g.get('D1')) for g in groups]

def extract_fixed_window(ch6, center_idx, window):
    half = window // 2
    center = int(center_idx)
    start = center - half
    end = start + window
    out = np.zeros((6, window), dtype=np.float32)
    src_lo = max(0, start)
    src_hi = min(ch6.shape[0], end)
    if src_hi > src_lo:
        dst_lo = src_lo - start
        out[:, dst_lo:dst_lo + (src_hi - src_lo)] = ch6[src_lo:src_hi].T
    return out

def build_labeled_cycle_dataset(entries, cfg):
    X_list, y_list, sid_list = [], [], []
    labeled_sids = set()
    for d0_path, d1_path, sid in entries:
        ann = load_annotations(sid)
        if ann is None:
            continue
        t0, ch0, t1, ch1 = load_session_signals(d0_path, d1_path)
        segs = maybe_align_annotation_time(ann['segments'], float(t0.min()), float(t0.max()))
        used_any = False
        for seg_start, seg_end, seg_label in segs:
            y_lab = map_to_supervised_class(seg_label)
            if y_lab is None:
                continue
            t_mid = 0.5 * (seg_start + seg_end)
            c0 = int(np.argmin(np.abs(t0 - t_mid)))
            c1 = int(np.argmin(np.abs(t1 - t_mid)))
            w0 = extract_fixed_window(ch0, c0, int(cfg['WINDOW']))
            w1 = extract_fixed_window(ch1, c1, int(cfg['WINDOW']))
            X_list.append(np.vstack([w0, w1]).astype(np.float32))
            y_list.append(CLASS_TO_IDX[y_lab])
            sid_list.append(sid)
            used_any = True
        if used_any:
            labeled_sids.add(sid)

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.int32)
    sids = np.array(sid_list, dtype=object)
    return X, y, sids, labeled_sids

def build_unlabeled_cycle_pool(entries, labeled_sids, cfg):
    X_u, meta = [], []
    for d0_path, d1_path, sid in entries:
        if sid in labeled_sids:
            continue
        t0, ch0, t1, ch1 = load_session_signals(d0_path, d1_path)
        peaks0 = detect_cycle_peaks(ch0[:, 3:6], cfg['FS'], cfg)
        peaks1 = detect_cycle_peaks(ch1[:, 3:6], cfg['FS'], cfg)
        pairs = merge_device_peaks_pairs(peaks0, peaks1, cfg['FS'], cfg['MERGE_GAP_S'])
        for p0, p1 in pairs:
            if p0 is not None and p1 is not None:
                t_ref = 0.5 * (t0[int(p0)] + t1[int(p1)])
            elif p0 is not None:
                t_ref = float(t0[int(p0)])
            else:
                t_ref = float(t1[int(p1)])
            c0 = int(np.argmin(np.abs(t0 - t_ref)))
            c1 = int(np.argmin(np.abs(t1 - t_ref)))
            w0 = extract_fixed_window(ch0, c0, int(cfg['WINDOW']))
            w1 = extract_fixed_window(ch1, c1, int(cfg['WINDOW']))
            X_u.append(np.vstack([w0, w1]).astype(np.float32))
            meta.append((sid, float(t_ref)))

    if not X_u:
        return np.zeros((0, 12, int(cfg['WINDOW'])), dtype=np.float32), []
    return np.stack(X_u).astype(np.float32), meta

def build_fourier_matrix(X_cycles, K=8):
    coeffs = np.fft.rfft(X_cycles, axis=2)
    mag = np.abs(coeffs[:, :, 1:K+1])
    return mag.reshape(len(X_cycles), -1).astype(np.float32)

def build_coherence_features(X_cycles, fs=50.0):
    n = len(X_cycles)
    out = np.zeros((n, 3), dtype=np.float32)
    nperseg = min(16, X_cycles.shape[2])
    for i in range(n):
        for ax in range(3):
            g0 = X_cycles[i, 3 + ax]
            g1 = X_cycles[i, 9 + ax]
            f, cxy = coherence(g0, g1, fs=fs, nperseg=nperseg)
            mask = (f >= 1.0) & (f <= 3.0)
            out[i, ax] = float(np.mean(cxy[mask])) if np.any(mask) else 0.0
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def compute_bio_features(cycle_matrix):
    cycle_matrix = np.asarray(cycle_matrix, dtype=np.float32)
    gz_mean_d0 = float(np.mean(cycle_matrix[5, :]))
    bio = np.array([gz_mean_d0], dtype=np.float32)
    return np.nan_to_num(bio, nan=0.0, posinf=0.0, neginf=0.0)

def build_feature_matrix(X_cycles, cfg):
    blocks = []

    if cfg.get('USE_RAW_FLAT_FEATURES', True):
        flat = X_cycles.reshape(len(X_cycles), -1).astype(np.float32)
        blocks.append(flat)

    if cfg.get('USE_FOURIER_FEATURES', False):
        blocks.append(build_fourier_matrix(X_cycles, K=int(cfg['FOURIER_K'])))

    if cfg.get('USE_BIOMECH_FEATURES', False):
        bio = np.stack([compute_bio_features(c) for c in X_cycles]).astype(np.float32)
        blocks.append(bio)

    if cfg.get('USE_COHERENCE_FEATURES', False):
        blocks.append(build_coherence_features(X_cycles, fs=float(cfg['FS'])))

    if not blocks:
        raise ValueError('At least one feature toggle must be enabled in CFG.')

    return np.nan_to_num(np.concatenate(blocks, axis=1), nan=0.0, posinf=0.0, neginf=0.0)

def build_signed_axis_features(X_cycles):
    out = np.zeros((len(X_cycles), 18), dtype=np.float32)
    for i, cm in enumerate(X_cycles):
        feats = []
        for ch in (3, 4, 5, 9, 10, 11):
            sig = cm[ch]
            feats.extend([
                float(np.mean(sig)),
                float(np.std(sig)),
                float(np.mean(np.sign(sig))),
            ])
        out[i] = np.array(feats, dtype=np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def build_cluster_purity_bundle(X_train_cycles, y_train_idx, X_unlabeled_cycles, cfg):
    if len(X_unlabeled_cycles) == 0:
        return None

    k = int(cfg.get('CLUSTER_K', 10))
    min_lr = int(cfg.get('MIN_LR_SAMPLES_PER_CLUSTER', 5))

    F_tr = build_signed_axis_features(X_train_cycles)
    F_unl = build_signed_axis_features(X_unlabeled_cycles)
    F_all = np.vstack([F_tr, F_unl])

    sc = StandardScaler()
    F_all_s = sc.fit_transform(F_all)
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    c_all = km.fit_predict(F_all_s)
    c_tr = c_all[:len(F_tr)]
    c_unl = c_all[len(F_tr):]

    dom_idx = np.full(k, -1, dtype=np.int32)
    purity = np.zeros(k, dtype=np.float32)
    n_labeled = np.zeros(k, dtype=np.int32)

    for c in range(k):
        mask = (c_tr == c)
        n = int(np.sum(mask))
        n_labeled[c] = n
        if n < min_lr:
            continue
        cnt = Counter(y_train_idx[mask].tolist())
        d, dc = cnt.most_common(1)[0]
        dom_idx[c] = int(d)
        purity[c] = float(dc / n)

    rows = []
    for c in range(k):
        rows.append({
            'cluster': c,
            'n_labeled': int(n_labeled[c]),
            'dominant_label_idx': int(dom_idx[c]) if dom_idx[c] >= 0 else None,
            'dominant_label': CLASS_NAMES[int(dom_idx[c])] if dom_idx[c] >= 0 else None,
            'purity': float(purity[c]),
        })
    diag_df = pd.DataFrame(rows).sort_values(['purity', 'n_labeled'], ascending=[False, False]).reset_index(drop=True)

    return {
        'clusters_unlabeled': c_unl,
        'dominant_label_idx_by_cluster': dom_idx,
        'purity_by_cluster': purity,
        'diag_df': diag_df,
    }

## 3) Model helpers (RF, PCA+GBM, 1D-CNN, BiLSTM)

In [34]:
def metric_row(model_name, y_train, y_pred_train, y_test, y_pred_test):
    tr_acc = accuracy_score(y_train, y_pred_train)
    te_acc = accuracy_score(y_test, y_pred_test)
    tr_f1 = f1_score(y_train, y_pred_train, average='macro', zero_division=0)
    te_f1 = f1_score(y_test, y_pred_test, average='macro', zero_division=0)
    return {
        'model': model_name,
        'train_error': float(1.0 - tr_acc),
        'test_error': float(1.0 - te_acc),
        'train_f1': float(tr_f1),
        'test_f1': float(te_f1),
        'train_accuracy': float(tr_acc),
        'test_accuracy': float(te_acc),
    }

def unavailable_row(model_name):
    return {
        'model': model_name,
        'train_error': np.nan,
        'test_error': np.nan,
        'train_f1': np.nan,
        'test_f1': np.nan,
        'train_accuracy': np.nan,
        'test_accuracy': np.nan,
    }

def train_rf(X_train_feat, y_train, X_test_feat, y_test):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train_feat)
    Xte = scaler.transform(X_test_feat)
    clf = RandomForestClassifier(
        n_estimators=400,
        max_depth=24,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42,
    )
    clf.fit(Xtr, y_train)
    row = metric_row('RF', y_train, clf.predict(Xtr), y_test, clf.predict(Xte))
    return row, clf, scaler

def train_pca_gbm(X_train_feat, y_train, X_test_feat, y_test, cfg):
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train_feat)
    Xte_s = scaler.transform(X_test_feat)
    pca = PCA(n_components=float(cfg.get('PCA_VAR', 0.99)), svd_solver='full')
    Xtr = pca.fit_transform(Xtr_s)
    Xte = pca.transform(Xte_s)
    clf = HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
    )
    clf.fit(Xtr, y_train)
    row = metric_row('PCA+GBM', y_train, clf.predict(Xtr), y_test, clf.predict(Xte))
    return row

def _train_nn_common(X_train, X_test):
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    Xtr = (X_train - mean) / std
    Xte = (X_test - mean) / std
    return Xtr.astype(np.float32), Xte.astype(np.float32)

def train_cnn(X_train, y_train, X_test, y_test, cfg):
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import TensorDataset, DataLoader
    except ImportError:
        return unavailable_row('1D-CNN')

    Xtr, Xte = _train_nn_common(X_train, X_test)
    n_classes = len(CLASS_NAMES)

    class CycleNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(12, 64, 5, padding=2),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Conv1d(64, 128, 5, padding=2),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))

        def forward(self, x):
            return self.head(self.conv(x))

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CycleNet().to(device)
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = torch.tensor(1.0 / (counts + 1.0), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    train_ds = TensorDataset(torch.tensor(Xtr), torch.tensor(y_train, dtype=torch.long))
    train_dl = DataLoader(train_ds, batch_size=int(cfg['NN_BATCH_SIZE']), shuffle=True)

    model.train()
    for _ in range(int(cfg['NN_EPOCHS'])):
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        yhat_tr = model(torch.tensor(Xtr).to(device)).argmax(dim=1).cpu().numpy().astype(np.int32)
        yhat_te = model(torch.tensor(Xte).to(device)).argmax(dim=1).cpu().numpy().astype(np.int32)
    return metric_row('1D-CNN', y_train, yhat_tr, y_test, yhat_te)

def train_bilstm(X_train, y_train, X_test, y_test, cfg):
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import TensorDataset, DataLoader
    except ImportError:
        return unavailable_row('BiLSTM')

    Xtr, Xte = _train_nn_common(X_train, X_test)
    Xtr = np.transpose(Xtr, (0, 2, 1))
    Xte = np.transpose(Xte, (0, 2, 1))
    n_classes = len(CLASS_NAMES)

    class BiLSTMNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size=12,
                hidden_size=64,
                num_layers=2,
                batch_first=True,
                bidirectional=True,
                dropout=0.3,
            )
            self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, n_classes))

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.head(out[:, -1, :])

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BiLSTMNet().to(device)
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = torch.tensor(1.0 / (counts + 1.0), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    train_ds = TensorDataset(torch.tensor(Xtr), torch.tensor(y_train, dtype=torch.long))
    train_dl = DataLoader(train_ds, batch_size=int(cfg['NN_BATCH_SIZE']), shuffle=True)

    model.train()
    for _ in range(int(cfg['NN_EPOCHS'])):
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        yhat_tr = model(torch.tensor(Xtr).to(device)).argmax(dim=1).cpu().numpy().astype(np.int32)
        yhat_te = model(torch.tensor(Xte).to(device)).argmax(dim=1).cpu().numpy().astype(np.int32)
    return metric_row('BiLSTM', y_train, yhat_tr, y_test, yhat_te)

def run_all_models(stage_name, X_train_cycles, y_train, X_test_cycles, y_test, cfg, return_rf=False):
    Xf_train = build_feature_matrix(X_train_cycles, cfg)
    Xf_test = build_feature_matrix(X_test_cycles, cfg)

    rows = []
    rf_row, rf_model, rf_scaler = train_rf(Xf_train, y_train, Xf_test, y_test)
    rows.append(rf_row)
    rows.append(train_pca_gbm(Xf_train, y_train, Xf_test, y_test, cfg))
    rows.append(train_bilstm(X_train_cycles, y_train, X_test_cycles, y_test, cfg))
    rows.append(train_cnn(X_train_cycles, y_train, X_test_cycles, y_test, cfg))

    out_df = pd.DataFrame(rows)[['model', 'train_error', 'test_error', 'train_f1', 'test_f1', 'train_accuracy', 'test_accuracy']]
    print(f'[{stage_name}]')
    print(out_df.to_string(index=False))

    payload = {'metrics_df': out_df}
    if return_rf:
        payload['rf_model'] = rf_model
        payload['rf_scaler'] = rf_scaler
    return payload

## 4) Build labeled dataset and fixed train/test split

In [35]:
sessions = discover_sessions(DATA_PROCESSED)
X_labeled, y_labeled, sid_labeled, labeled_sid_set = build_labeled_cycle_dataset(sessions, CFG)

idx = np.arange(len(y_labeled))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y_labeled, random_state=42)

X_train_lab = X_labeled[tr_idx]
y_train_lab = y_labeled[tr_idx]
X_test_lab = X_labeled[te_idx]
y_test_lab = y_labeled[te_idx]

## 5) Initial models (labeled-only)

In [36]:
initial_bundle = run_all_models(
    'INITIAL (labeled only)',
    X_train_lab,
    y_train_lab,
    X_test_lab,
    y_test_lab,
    CFG,
    return_rf=True,
)
initial_results_df = initial_bundle['metrics_df']

[INITIAL (labeled only)]
  model  train_error  test_error  train_f1  test_f1  train_accuracy  test_accuracy
     RF     0.000000    0.161417  1.000000 0.857966        1.000000       0.838583
PCA+GBM     0.000000    0.183071  1.000000 0.840899        1.000000       0.816929
 BiLSTM     0.012821    0.185039  0.989625 0.823850        0.987179       0.814961
 1D-CNN     0.073471    0.179134  0.951058 0.842000        0.926529       0.820866


## 6) Pseudo-label unlabeled data using initial RF

In [37]:
X_unlabeled, unlabeled_meta = build_unlabeled_cycle_pool(sessions, labeled_sid_set, CFG)
cluster_diag_df = pd.DataFrame()

if len(X_unlabeled) > 0:
    Xf_unlabeled = build_feature_matrix(X_unlabeled, CFG)
    Xu_s = initial_bundle['rf_scaler'].transform(Xf_unlabeled)
    probs = initial_bundle['rf_model'].predict_proba(Xu_s)
    pred_idx = np.argmax(probs, axis=1).astype(np.int32)
    pred_conf = np.max(probs, axis=1)
    base_keep = pred_conf >= float(CFG['PSEUDO_CONF_THRESHOLD'])

    cluster_id = np.full(len(X_unlabeled), -1, dtype=np.int32)
    cluster_purity = np.zeros(len(X_unlabeled), dtype=np.float32)
    cluster_dom_idx = np.full(len(X_unlabeled), -1, dtype=np.int32)
    rf_cluster_agree = np.zeros(len(X_unlabeled), dtype=bool)

    if bool(CFG.get('USE_CLUSTER_PURITY_GATE', False)):
        cluster_bundle = build_cluster_purity_bundle(X_train_lab, y_train_lab, X_unlabeled, CFG)
        if cluster_bundle is not None:
            cluster_diag_df = cluster_bundle['diag_df']
            cluster_id = cluster_bundle['clusters_unlabeled']
            dom_by_c = cluster_bundle['dominant_label_idx_by_cluster']
            pur_by_c = cluster_bundle['purity_by_cluster']

            cluster_dom_idx = np.array([int(dom_by_c[c]) for c in cluster_id], dtype=np.int32)
            cluster_purity = np.array([float(pur_by_c[c]) for c in cluster_id], dtype=np.float32)

            has_dom = cluster_dom_idx >= 0
            purity_ok = has_dom & (cluster_purity >= float(CFG['CLUSTER_PURITY_THRESHOLD']))
            rf_cluster_agree = purity_ok & (pred_idx == cluster_dom_idx)

            if bool(CFG.get('REQUIRE_RF_CLUSTER_AGREEMENT', True)):
                keep = base_keep & rf_cluster_agree
            else:
                keep = base_keep & purity_ok
        else:
            keep = base_keep
    else:
        keep = base_keep

    inference_df = pd.DataFrame({
        'session_id': [m[0] for m in unlabeled_meta],
        't_mid_s': [m[1] for m in unlabeled_meta],
        'pred_label_idx': pred_idx,
        'pred_label': [CLASS_NAMES[i] for i in pred_idx],
        'pred_confidence': pred_conf,
        'cluster_id': cluster_id,
        'cluster_purity': cluster_purity,
        'cluster_dominant_label_idx': cluster_dom_idx,
        'cluster_dominant_label': [CLASS_NAMES[i] if i >= 0 else None for i in cluster_dom_idx],
        'rf_cluster_agree': rf_cluster_agree,
        'keep_pseudo': keep,
    })
    for i, cls in enumerate(CLASS_NAMES):
        inference_df[f'prob_{cls}'] = probs[:, i]

    X_pseudo = X_unlabeled[keep]
    y_pseudo = pred_idx[keep]
else:
    inference_df = pd.DataFrame(columns=[
        'session_id', 't_mid_s', 'pred_label_idx', 'pred_label', 'pred_confidence',
        'cluster_id', 'cluster_purity', 'cluster_dominant_label_idx', 'cluster_dominant_label',
        'rf_cluster_agree', 'keep_pseudo'
    ])
    X_pseudo = np.zeros((0, 12, int(CFG['WINDOW'])), dtype=np.float32)
    y_pseudo = np.zeros((0,), dtype=np.int32)

## 7) Final models (labeled + pseudo-labeled)

In [38]:
X_train_final = np.concatenate([X_train_lab, X_pseudo], axis=0)
y_train_final = np.concatenate([y_train_lab, y_pseudo], axis=0)

final_bundle = run_all_models(
    'FINAL (labeled + pseudo-labeled)',
    X_train_final,
    y_train_final,
    X_test_lab,
    y_test_lab,
    CFG,
    return_rf=False,
)
final_results_df = final_bundle['metrics_df']

KeyboardInterrupt: 

## 8) In-memory artifacts

- `initial_results_df`, `final_results_df`: metric tables for all 4 models
- `inference_df`: unlabeled predictions + class probabilities + cluster-gate columns
- `cluster_diag_df`: cluster -> dominant label/purity diagnostics
- `X_pseudo`, `y_pseudo`: selected pseudo-labeled cycles
- Feature toggles are controlled in `CFG`.

## 9) Train/Test size summary

In [39]:
size_df = pd.DataFrame([
    {
        'stage': 'INITIAL (labeled only)',
        'applies_to': 'All classifiers',
        'train_samples': int(len(X_train_lab)),
        'test_samples': int(len(X_test_lab)),
    },
    {
        'stage': 'FINAL (labeled + pseudo-labeled)',
        'applies_to': 'All classifiers',
        'train_samples': int(len(X_train_final)),
        'test_samples': int(len(X_test_lab)),
    },
])
print(size_df.to_string(index=False))

                           stage      applies_to  train_samples  test_samples
          INITIAL (labeled only) All classifiers           2028           508
FINAL (labeled + pseudo-labeled) All classifiers           2028           508
